# Silver_Layer - Cleaning & Enrichment

1. Read Bronze tables (`bronze_customers`, `bronze_devices`, `bronze_service_jobs`).
2. Remove duplicate job records on `job_id`.
3. Fix blank `technician_name` values using a lookup table generated from matching `technician_id` values.
4. Replace missing `repair_notes` with a default message.
5. Ensure `completed_date` and `actual_cost` remain NULL for non-completed jobs.
6. Convert date columns (like `received_date`, `promised_date`, `completed_date`, and `registration_date`) to proper Date types.
7. Create a new column named `Repair_Duration` representing the number of days a repair took.
8. Join the three datasets together.
9. Save the result as a single enriched Delta table named `silver_enriched_jobs`.

### Load Shared Configuration and Helper Functions
Loads config values and reusable functions (`save_as_delta`, `convert_dates`, `logger`) from `00_Config_Utils`, so this notebook doesn't repeat code already written elsewhere.

In [0]:
%run ./00_Config_Utils

# logging - Shared Configuration and Helper Functions

This keeps the project **modular** (reusable functions instead of copy-pasted code) and **parameterized** (thresholds and table names live in one place, not hardcoded everywhere).

### Step 1: Set Up Logging
We use Python's built-in `logging` module instead of plain `print()` statements. This gives every message a timestamp and a severity level (INFO, WARNING, ERROR), which is standard practice in real data pipelines.

2026-07-11 19:06:12,305 - INFO - Logger initialized for ServiceTrack pipeline.


### Step 2: Configuration Values (Parameterized Settings)
All table names, file paths, and business thresholds live here. If any of these need to change later (e.g. a different delay threshold), we only edit this one place — nothing else in the pipeline needs to change.

2026-07-11 19:06:12,642 - INFO - Configuration values loaded.


### Step 3: Reusable Function - Load a CSV File
Wraps CSV reading in a function with error handling, so every notebook that needs to read a CSV can call this one function instead of repeating the same code.

### Step 4: Reusable Function - Save a DataFrame as a Delta Table
Every layer (Bronze, Silver, Gold) saves DataFrames as Delta tables the same way. This function avoids repeating that logic in every notebook.

### Step 5: Reusable Function - Convert Multiple Columns to Date Type
Used in the Silver layer to convert received_date, promised_date, completed_date, and registration_date in one call instead of writing a separate withColumn() line for each.

### Step 6: Reusable Function - Validate Row Counts
Used by the End-to-End Pipeline notebook to check that row counts match expected values, and log a clear Success/Warning message.

### Import PySpark Functions

In [0]:
# Import standard Spark SQL functions
from pyspark.sql.functions import col, to_date, datediff, lit, coalesce, when

###  Read Bronze Delta Tables

In [0]:
# Load Bronze tables using config table names, wrapped in error handling
try:
    df_customers = spark.read.table(BRONZE_CUSTOMERS_TABLE)
    df_devices = spark.read.table(BRONZE_DEVICES_TABLE)
    df_jobs = spark.read.table(BRONZE_SERVICE_JOBS_TABLE)
    logger.info("Bronze tables loaded successfully into Silver notebook.")
except Exception as e:
    logger.error(f"Failed to read Bronze tables: {e}")
    raise

2026-07-11 19:06:14,604 - INFO - Bronze tables loaded successfully into Silver notebook.


### Remove Duplicate Job Records

In [0]:
# Drop duplicate records based on job_id column
df_jobs_dedup = df_jobs.dropDuplicates(["job_id"])

# Check record counts before and after deduplication to verify
print(f"Raw jobs count: {df_jobs.count()}")
print(f"Deducted jobs count: {df_jobs_dedup.count()}")

Raw jobs count: 1510
Deducted jobs count: 1500


###Handle Blank Technician Names


In [0]:
# Build a lookup DataFrame 
df_tech_lookup = df_jobs_dedup.select("technician_id", "technician_name") \
    .filter((col("technician_name").isNotNull()) & (col("technician_name") != "")) \
    .distinct()

# Drop the original technician_name column
df_jobs_no_tech_name = df_jobs_dedup.drop("technician_name")

# Join the jobs DataFrame with our technician lookup DataFrame
df_jobs_with_tech = df_jobs_no_tech_name.join(df_tech_lookup, on="technician_id", how="left")

###Check for Inconsistent Technician Names


In [0]:
# Check if any technician_id has more than one distinct technician_name
df_tech_conflicts = df_tech_lookup.groupBy("technician_id").count().filter(col("count") > 1)

conflict_count = df_tech_conflicts.count()
if conflict_count > 0:
    print(f"Warning: {conflict_count} technician_id(s) have more than one name. Review below:")
    display(df_tech_conflicts)
else:
    print("Success: Every technician_id maps to exactly one technician_name. Safe to join.")

Success: Every technician_id maps to exactly one technician_name. Safe to join.


### Convert Date Columns and Create Repair_Duration

### Raw Date Format Before Converting


In [0]:
# Display sample raw date values before converting, to confirm the format is yyyy-MM-dd
display(df_jobs_with_tech.select("received_date", "promised_date", "completed_date").limit(5))

2026-07-11 19:06:21,061 - INFO - Received command c on object id p0


received_date,promised_date,completed_date
2024-01-29,2024-02-03,2024-01-31
2024-02-29,2024-03-05,2024-03-07
2024-03-13,2024-03-18,2024-03-17
2024-01-23,2024-01-28,null
2024-01-28,2024-02-02,2024-01-31


In [0]:
# Convert dates using the shared convert_dates() helper instead of repeating withColumn() for each column
df_jobs_dates = convert_dates(df_jobs_with_tech, ["received_date", "promised_date", "completed_date"])

# Create Repair_Duration column
df_jobs_duration = df_jobs_dates.withColumn("Repair_Duration", datediff(col("completed_date"), col("received_date")))

2026-07-11 19:06:31,040 - INFO - Received command c on object id p0
2026-07-11 19:06:31,124 - INFO - Converted columns to Date type: ['received_date', 'promised_date', 'completed_date']


###Add job_status_flag Column (Delayed / On Time / Not Completed)


In [0]:
# Create job_status_flag: Delayed / On Time / Not Completed
df_jobs_duration = df_jobs_duration.withColumn(
    "job_status_flag",
    when(col("completed_date").isNull(), "Not Completed")
    .when(col("completed_date") > col("promised_date"), "Delayed")
    .otherwise("On Time")
)

# Quick check: see the distribution of the new flag
display(df_jobs_duration.groupBy("job_status_flag").count())

2026-07-11 19:06:31,763 - INFO - Received command c on object id p0


job_status_flag,count
On Time,833
Delayed,297
Not Completed,370


### Clean Repair Notes and Retain Incomplete Job Nulls


In [0]:
# Use fillna to replace NULL repair notes with a default string
df_jobs_cleaned = df_jobs_duration.fillna({"repair_notes": "No notes provided"})

2026-07-11 19:06:33,918 - INFO - Received command c on object id p0


###Clean and Prepare Dimension Tables (Customers and Devices)


In [0]:
# Convert customers registration_date using the same shared helper for consistency
df_customers_cleaned = convert_dates(df_customers, ["registration_date"])

df_devices_cleaned = df_devices

2026-07-11 19:06:34,222 - INFO - Converted columns to Date type: ['registration_date']


### Join Datasets to Create Enriched Fact Table

In [0]:
# Join jobs with customers and devices, wrapped in error handling
try:
    df_joined_cust = df_jobs_cleaned.join(df_customers_cleaned, on="customer_id", how="left")
    df_enriched = df_joined_cust.join(df_devices_cleaned, on="device_id", how="left")
    logger.info(f"Join complete. Enriched row count: {df_enriched.count()}")
except Exception as e:
    logger.error(f"Failed to join Silver datasets: {e}")
    raise

display(df_enriched.limit(5))

2026-07-11 19:06:36,790 - INFO - Join complete. Enriched row count: 1500


device_id,customer_id,technician_id,job_id,issue_type,job_status,received_date,promised_date,completed_date,repair_notes,estimated_cost,actual_cost,technician_name,Repair_Duration,job_status_flag,customer_name,phone_number,email,city,registration_date,brand,device_type,model_series,warranty_months,price_range
DEV019,CUST0295,T005,JOB01181,Battery Issue,Completed,2024-01-29,2024-02-03,2024-01-31,Customer informed,4775.15,6845.21,Kavitha Nair,2,On Time,Meera Khan,9995866934,meera.khan51@outlook.com,Pune,2023-04-09,Lenovo,Printer,Lenovo PRI-Series,12,Premium (> 40K)
DEV035,CUST0108,T006,JOB00070,Keyboard Fault,Completed,2024-02-29,2024-03-05,2024-03-07,No notes provided,1450.68,4850.57,Deepak Sharma,7,Delayed,Rahul Bajaj,9423573322,rahul.bajaj10@outlook.com,Bhopal,2022-04-14,Motorola,Laptop,Motorola LAP-Series,24,Mid-range (15K-40K)
DEV020,CUST0001,T007,JOB00991,Software Crash,Cancelled,2024-03-13,2024-03-18,2024-03-17,Spare part ordered,617.6,null,Meena Reddy,4,On Time,Kiran Patel,9433218196,kiran.patel5@gmail.com,Delhi,2022-08-12,Lenovo,Smartphone,Lenovo SMA-Series,12,Budget (< 15K)
DEV022,CUST0273,T008,JOB01238,Hard Disk Failure,Pending,2024-01-23,2024-01-28,null,Returned to customer,3234.8,null,Arjun Iyer,null,Not Completed,Shivam Singh,9074329271,shivam.singh17@hotmail.com,Lucknow,2023-03-16,Asus,Refrigerator,Asus REF-Series,18,Budget (< 15K)
DEV039,CUST0152,T005,JOB00956,Overheating,Completed,2024-01-28,2024-02-02,2024-01-31,Returned to customer,5936.46,4845.37,Kavitha Nair,3,On Time,Nisha Gupta,9874315274,nisha.gupta92@yahoo.com,Mumbai,2022-12-12,Realme,Laptop,Realme LAP-Series,6,Premium (> 40K)


### Save as Silver Delta Table

In [0]:
# Save using the shared save_as_delta() helper
save_as_delta(df_enriched, SILVER_ENRICHED_JOBS_TABLE, overwrite_schema=True)

2026-07-11 19:06:46,120 - INFO - Saved Delta table: silver_enriched_jobs
